In [ ]:
"""
Consumer Psychology Pipeline for Alternative Proteins
Full Pipeline: Zero-shot Classification + BERTopic + KeyBERT + Statistical & Construct Analysis
Includes:
  - Top discriminative keywords per construct (Option 1)
  - Mechanism → Intervention mapping table (Option 2)
"""

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import mannwhitneyu
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

try:
    from keybert import KeyBERT
    KEYBERT_AVAILABLE = True
except ImportError:
    KEYBERT_AVAILABLE = False
    print("KeyBERT not installed. Install with: pip install keybert")


# 1. CONFIGURATION & FILE PATHS

CSV_PATH = r"data\alt_protein_discussions.csv"
OUTPUT_PATH = r"data\alt_protein_results.csv"
SUMMARY_PATH = r"data\alt_protein_summary.csv"

# 2. CONCEPTUAL TAXONOMY & HYPOTHESES

DISPLAY_LABELS = [
    "differentiation_from_meat",
    "perceived_processing",
    "naturalness_perceptions",
    "taste_expectations",
    "sustainability_differentiation",
    "anchoring_to_meat",
    "perceived_loss_switching",
    "trust_and_credibility",
    "uncertainty_and_risk",
    "social_acceptance",
]

HYPOTHESIS_LABELS = [
    "The consumer focuses on how this alternative product differs from or compares unfavorably to conventional meat.",
    "The consumer expresses concern over ultra-processing, heavy additives, or industrial chemical manufacturing.",
    "The consumer perceives this alternative protein as artificial, synthetic, fake, or unnatural.",
    "The consumer expects unappealing taste, rubbery texture, or poor sensory quality compared to meat.",
    "The consumer evaluates or questions the environmental, climate, and sustainability claims of the product.",
    "The consumer uses real conventional meat as the default benchmark and resists changing established eating habits.",
    "The consumer fears losing enjoyment, satiety, or nutritional value when replacing conventional meat.",
    "The consumer doubts producer transparency, corporate motivations, or clean labeling.",
    "The consumer expresses uncertainty regarding long-term health safety, digestibility, or unknown ingredients.",
    "The consumer worries about social judgment, family rejection, or violating cultural dining traditions.",
]

LABEL_MAP = dict(zip(DISPLAY_LABELS, HYPOTHESIS_LABELS))

def safe_bar(value):
    if pd.isna(value) or np.isnan(value):
        return " " * 20
    val = int(value)
    return "█" * val + "░" * (20 - val) if val <= 20 else "█" * 20

# 3. DATA LOADING

print("=" * 70)
print("   ALTERNATIVE PROTEIN CONSUMER PSYCHOLOGY PIPELINE")
print("   Proof-of-Concept for Study 1: Computational Analysis of Consumer Discourse")
print("=" * 70)

print("\n[1] Loading data...")
df = pd.read_csv(CSV_PATH)
print(f"    Loaded {len(df)} alternative-protein consumer discussions.")


df['platform'] = df['source'].apply(lambda x: 'Twitter' if x == 'X' else 'Reddit')

texts = df["text"].dropna().astype(str).tolist()  # using column 'text'
print(f"    {len(texts)} valid texts ready for computational analysis.")

# 4. ZERO-SHOT CLASSIFIER INITIALIZATION

print("\n[2] Loading zero-shot NLI classifier (BART-Large-MNLI)...")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    return_all_scores=True,
    device=0,
    batch_size=8,
)
print("    Classifier loaded successfully.")
print("\n    Target Constructs:")
for i, label in enumerate(DISPLAY_LABELS):
    print(f"      {i+1}. {label:<30}: {HYPOTHESIS_LABELS[i]}")

def predict_construct_probabilities(texts_batch):
    if isinstance(texts_batch, str):
        texts_batch = [texts_batch]
    results = classifier(texts_batch, HYPOTHESIS_LABELS, multi_label=True)
    if isinstance(results, dict):
        results = [results]
    probs = []
    for res in results:
        label_scores = {label: score for label, score in zip(res["labels"], res["scores"])}
        probs.append([label_scores[hyp] for hyp in HYPOTHESIS_LABELS])
    return np.array(probs)

# 5. BATCH CLASSIFICATION EXECUTION

print("\n[3] Executing zero-shot classification on alternative protein corpus...")
batch_size = 16
all_probs = []
total_batches = (len(texts) + batch_size - 1) // batch_size

for i in range(0, len(texts), batch_size):
    batch = texts[i:i + batch_size]
    probs = predict_construct_probabilities(batch)
    all_probs.append(probs)
    print(f"    Processed batch {i//batch_size + 1}/{total_batches}")

probs_array = np.vstack(all_probs)
print(f"    Classification complete for {len(texts)} texts.")

for i, col in enumerate(DISPLAY_LABELS):
    df[col] = probs_array[:, i]


# 6. AGGREGATE REPORT & DOMINANT BARRIERS

mean_probs = probs_array.mean(axis=0)
std_probs = probs_array.std(axis=0)

print("\n" + "=" * 70)
print(" AGGREGATE PSYCHOLOGICAL BARRIER REPORT (ALTERNATIVE PROTEINS)")
print("=" * 70)
print(f"\nOverall Construct Prevalence across texts (n = {len(texts)}):")
print("-" * 65)

for i, label in enumerate(DISPLAY_LABELS):
    mean_pct = mean_probs[i] * 100
    std_pct = std_probs[i] * 100
    bar = safe_bar(mean_pct / 5)
    print(f"  {label:<32}: {mean_pct:5.1f}%  (± {std_pct:.1f}%)  {bar}")

# Significant barriers (>40%)
print("\n[4] Significant Psychological Barriers (>40% prevalence):")
for label, score in zip(DISPLAY_LABELS, mean_probs):
    if score > 0.4:
        print(f"  ✅ {label}: {score*100:.1f}%")

# Dominant construct per text
dominant_indices = np.argmax(probs_array, axis=1)
df["dominant_construct"] = [DISPLAY_LABELS[idx] for idx in dominant_indices]
dominant_counts = df["dominant_construct"].value_counts()

print("\n" + "-" * 65)
print("Dominant Construct Distribution across Corpus:")
for construct, count in dominant_counts.items():
    percentage = (count / len(texts)) * 100
    print(f"  {construct:<32}: {count:>3} texts ({percentage:.1f}%)")

  
# 7. CROSS-PLATFORM COMPARISON (Optional)
  
if 'platform' in df.columns:
    print("\n[5] Cross-Platform Comparison (Twitter vs Reddit):")
    twitter_data = df[df['platform'] == 'Twitter']
    reddit_data = df[df['platform'] == 'Reddit']
    
    for label in DISPLAY_LABELS:
        if len(twitter_data) > 0 and len(reddit_data) > 0:
            stat, p = mannwhitneyu(twitter_data[label], reddit_data[label])
            if p < 0.05:
                print(f"  {label}: Significant difference (p = {p:.4f})")

  
# 8. EXPLANABLE AI EVIDENCE EXTRACTION (KeyBERT)
  
print("\n[6] Generating interpretable keyword attribution with KeyBERT...")
if KEYBERT_AVAILABLE:
    kw_model = KeyBERT()

sample_indices = np.random.choice(len(texts), min(3, len(texts)), replace=False)

print("\n" + "=" * 70)
print(" SAMPLE TEXT EVIDENCE & CONSTRUCT ATTRIBUTION")
print("=" * 70)

for idx in sample_indices:
    text = texts[idx]
    print(f"\nSAMPLE TEXT:\n'{text}'\n")
    
    scores = df.loc[idx, DISPLAY_LABELS].values.astype(float)
    dom_idx = np.argmax(scores)
    dom_label = DISPLAY_LABELS[dom_idx]
    
    print("Construct Probabilities:")
    for i, label in enumerate(DISPLAY_LABELS):
        pct = scores[i] * 100
        print(f"  {label:<30}: {pct:5.1f}% {safe_bar(pct/5)}")
    
    print(f"\n  DOMINANT BARRIER: {dom_label.upper()} ({scores[dom_idx]*100:.1f}%)")
    
    if KEYBERT_AVAILABLE:
        keywords = kw_model.extract_keywords(
            text, keyphrase_ngram_range=(1, 2), stop_words="english", top_n=4
        )
        print("  Extracted Supporting Keyphrases:")
        for kw, score in keywords:
            print(f"    • '{kw}' (semantic relevance: {score:.3f})")
    print("-" * 50)

  
# 9. BERTopic THEME EXTRACTION 
  
print("\n[7] Extracting thematic patterns with BERTopic...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
topic_model = BERTopic(embedding_model=embedding_model, nr_topics="auto", verbose=True)
topics, probs = topic_model.fit_transform(texts)
topic_info = topic_model.get_topic_info()
print("\nDiscovered Themes in Alternative-Protein Discourse:")
print(topic_info[['Topic', 'Count', 'Name']].head(10))

  
# 10. TOP DISCRIMINATIVE KEYWORDS PER CONSTRUCT
  
print("\n[8] Top Discriminative Keywords for Each Psychological Construct:")
print("=" * 70)

if KEYBERT_AVAILABLE and 'dominant_construct' in df.columns:
    for label in DISPLAY_LABELS:
        # Get texts where this construct is dominant
        dominant_texts = df[df['dominant_construct'] == label]['text'].tolist()
        if len(dominant_texts) >= 3:
            # Sample up to 5 texts
            sample_texts = dominant_texts[:5]
            # Extract keywords
            keywords = []
            for text in sample_texts:
                try:
                    kw = kw_model.extract_keywords(
                        text, keyphrase_ngram_range=(1,2), stop_words='english', top_n=3
                    )
                    keywords.extend([k[0] for k in kw])
                except:
                    continue
            # Get top 5 unique keywords (preserve order)
            unique_kw = []
            for k in keywords:
                if k not in unique_kw:
                    unique_kw.append(k)
            unique_kw = unique_kw[:5]
            if unique_kw:
                print(f"\n  {label.upper()} (n={len(dominant_texts)} texts):")
                print(f"    → {', '.join(unique_kw)}")
        else:
            print(f"\n  {label.upper()}: insufficient texts for keyword extraction")

  
# 11. MECHANISM → INTERVENTION MAPPING (DYNAMIC)
  
print("\n" + "=" * 70)
print(" PSYCHOLOGICAL MECHANISM → INTERVENTION MAPPING")
print("=" * 70)

# Define intervention strategies for each construct
intervention_map = {
    "uncertainty_and_risk": "Third-party safety certifications; transparent ingredient disclosure; independent lab results",
    "differentiation_from_meat": "Positive framing of unique attributes; avoid direct comparison where product falls short",
    "naturalness_perceptions": "Reduce packaging colour saturation (Steiner et al., 2026); emphasise plant-origin ingredients",
    "perceived_processing": "Highlight simple, transparent processing methods; reframe as natural fermentation or food craft",
    "sustainability_differentiation": "Pair sustainability claims with taste and quality assurances (Kunz et al., 2021)",
    "taste_expectations": "Sensory-rich descriptions; chef-led culinary preparation tips",
    "perceived_loss_switching": "Emphasise satisfaction, satiety, and nutritional adequacy",
    "trust_and_credibility": "Independent third-party certifications; transparent LCA data",
    "social_acceptance": "Highlight growing adoption; peer testimonials; cultural fit",
    "anchoring_to_meat": "Frame as complementary addition; flexitarian integration"
}


sorted_for_table = sorted(zip(DISPLAY_LABELS, mean_probs), key=lambda x: x[1], reverse=True)


top_constructs = sorted_for_table[:5]


print("\n  +-----------------------------+--------------------------------------------------+")
print("  | Psychological Mechanism     | Potential Intervention                           |")
print("  +-----------------------------+--------------------------------------------------+")

for label, score in top_constructs:
   
    display_name = label.replace('_', ' ').title()
    percentage = score * 100
    
    
    intervention = intervention_map.get(label, "Provide targeted informational framing addressing consumer risk")
    
    
    words = intervention.split()
    lines = []
    current_line = ""
    for word in words:
        if len(current_line) + len(word) + 1 <= 48:
            current_line += " " + word if current_line else word
        else:
            lines.append(current_line)
            current_line = word
    if current_line:
        lines.append(current_line)
    
    # Print the construct line
    print(f"  | {display_name:<27} | {lines[0]:<48} |")
    
    
    for line in lines[1:]:
        print(f"  | {'':27} | {line:<48} |")
    
    print("  +-----------------------------+--------------------------------------------------+")

  
# 12. INTERVENTION RECOMMENDATIONS 
  
print("\n" + "=" * 70)
print(" PSYCHOLOGICALLY TAILORED INTERVENTION RECOMMENDATIONS (STUDY 3)")
print("=" * 70)

interventions = {
    "differentiation_from_meat": (
        "• Apply positive differentiation framing (emphasize unique culinary"
        " strengths rather than mimicking meat perfectly).\n• Avoid inviting"
        " direct organoleptic comparison where the product falls short."
    ),
    "perceived_processing": (
        "• Highlight ingredient transparency and simple processing"
        " techniques.\n• Reframe processing steps in terms of natural"
        " fermentation, traditional cooking, or wholesome food craft."
    ),
    "naturalness_perceptions": (
        "• Reduce visual package color saturation (Steiner et al., 2026) to"
        " enhance perceived naturalness.\n• Emphasize plant-origin"
        " ingredients and minimal artificial additives."
    ),
    "taste_expectations": (
        "• Utilize sensory-rich descriptions focusing on umami, texture, and"
        " mouthfeel.\n• Provide chef-led culinary preparation tips to counter"
        " negative taste expectations."
    ),
    "sustainability_differentiation": (
        "• Pair environmental claims with explicit quality and taste assurances"
        " to eliminate the 'sustainability liability' (Kunz et al., 2021)."
    ),
    "anchoring_to_meat": (
        "• Frame products as complementary additions to flexitarian diets"
        " rather than demanding immediate total replacement of meat."
    ),
    "trust_and_credibility": (
        "• Leverage independent third-party certifications and transparent"
        " life-cycle assessment (LCA) data."
    ),
}

sorted_constructs = sorted(
    zip(DISPLAY_LABELS, mean_probs), key=lambda x: x[1], reverse=True
)

print("\nTop Identified Psychological Barriers & Targeted Communication Strategy:")
for label, score in sorted_constructs[:3]:
    print(f"\n[{label.upper()}] (Prevalence: {score*100:.1f}%)")
    if label in interventions:
        print(interventions[label])
    else:
        print("• Provide targeted informational framing addressing consumer risk.")

  
# 13. SAVE RESULTS
  
df.to_csv(OUTPUT_PATH, index=False)
summary_df = pd.DataFrame({
    "construct": DISPLAY_LABELS,
    "mean_probability_pct": mean_probs * 100,
    "std_probability_pct": std_probs * 100,
})
summary_df.to_csv(SUMMARY_PATH, index=False)

print("\n" + "=" * 70)
print(f"[✔] Analysis Complete. Results saved to: {OUTPUT_PATH}")
print("=" * 70)

   ALTERNATIVE PROTEIN CONSUMER PSYCHOLOGY PIPELINE
   Proof-of-Concept for Study 1: Computational Analysis of Consumer Discourse

[1] Loading data...
    Loaded 123 alternative-protein consumer discussions.
    123 valid texts ready for computational analysis.

[2] Loading zero-shot NLI classifier (BART-Large-MNLI)...


Device set to use cuda:0


    Classifier loaded successfully.

    Target Constructs:
      1. differentiation_from_meat     : The consumer focuses on how this alternative product differs from or compares unfavorably to conventional meat.
      2. perceived_processing          : The consumer expresses concern over ultra-processing, heavy additives, or industrial chemical manufacturing.
      3. naturalness_perceptions       : The consumer perceives this alternative protein as artificial, synthetic, fake, or unnatural.
      4. taste_expectations            : The consumer expects unappealing taste, rubbery texture, or poor sensory quality compared to meat.
      5. sustainability_differentiation: The consumer evaluates or questions the environmental, climate, and sustainability claims of the product.
      6. anchoring_to_meat             : The consumer uses real conventional meat as the default benchmark and resists changing established eating habits.
      7. perceived_loss_switching      : The consumer fears 

2026-08-13 00:52:34,801 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

2026-08-13 00:52:35,126 - BERTopic - Embedding - Completed ✓
2026-08-13 00:52:35,128 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-13 00:52:57,841 - BERTopic - Dimensionality - Completed ✓
2026-08-13 00:52:57,843 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-13 00:52:57,863 - BERTopic - Cluster - Completed ✓
2026-08-13 00:52:57,866 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-13 00:52:57,899 - BERTopic - Representation - Completed ✓
2026-08-13 00:52:57,901 - BERTopic - Topic reduction - Reducing number of topics
2026-08-13 00:52:57,914 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-13 00:52:57,943 - BERTopic - Representation - Completed ✓
2026-08-13 00:52:57,947 - BERTopic - Topic reduction - Reduced number of topics from 3 to 3



Discovered Themes in Alternative-Protein Discourse:
   Topic  Count                  Name
0     -1     21   -1_it_and_plant_the
1      0     54   0_the_and_burger_it
2      1     48  1_meat_grown_lab_the

[8] Top Discriminative Keywords for Each Psychological Construct:

  DIFFERENTIATION_FROM_MEAT (n=27 texts):
    → meat environmentally, grown meat, animal meat, meat environmental, meat biologically

  PERCEIVED_PROCESSING: insufficient texts for keyword extraction

  NATURALNESS_PERCEPTIONS (n=8 texts):
    → taste nutrients, nutrients feeling, taste, burger great, burger

  TASTE_EXPECTATIONS: insufficient texts for keyword extraction

  SUSTAINABILITY_DIFFERENTIATION (n=18 texts):
    → grown meat, meat meat, meat, natural meat, lab grown

  ANCHORING_TO_MEAT: insufficient texts for keyword extraction

  PERCEIVED_LOSS_SWITCHING (n=4 texts):
    → veggie meats, love veggie, meat vegetarian, steak, steak love

  TRUST_AND_CREDIBILITY: insufficient texts for keyword extraction

  U